In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp
from scipy.optimize import minimize

df = pd.read_csv("../data/uk_final.csv")
df["date"] = pd.to_datetime(df["date"])

df_sir = df[(df["date"] >= "2021-12-01") & (df["date"] <= "2022-02-01")].copy()
df_sir = df_sir.reset_index(drop=True)

plt.figure(figsize=(10, 4))
plt.plot(df_sir["date"], df_sir["cases_7d"], label="Observed 7-day average cases")
plt.title("Selected modelling window")
plt.xlabel("Date")
plt.ylabel("Cases")
plt.legend()
plt.grid()
plt.show()

In [ ]:
y_data = df_sir["cases_7d"].values
t_data = np.arange(len(df_sir))

N = 56_000_000

I0 = max(float(y_data[0]) * 7, 1.0)
R0 = 0.0
S0 = N - I0 - R0

print("Number of time points:", len(t_data))
print("Initial observed cases:", y_data[0])
print("S0 =", S0)
print("I0 =", I0)
print("R0 =", R0)

In [ ]:
def sir_model_constant(t, y, beta, gamma, N):
    S, I, R = y
    dSdt = -beta * S * I / N
    dIdt = beta * S * I / N - gamma * I
    dRdt = gamma * I
    return [dSdt, dIdt, dRdt]

def simulate_sir_constant(beta, gamma, N, S0, I0, R0, t_eval):
    y0 = [S0, I0, R0]

    sol = solve_ivp(
        fun=lambda t, y: sir_model_constant(t, y, beta, gamma, N),
        t_span=(t_eval[0], t_eval[-1]),
        y0=y0,
        t_eval=t_eval,
        method="RK45"
    )

    S = sol.y[0]
    I = sol.y[1]
    R = sol.y[2]
    incidence = beta * S * I / N

    return S, I, R, incidence

def loss_constant(params):
    beta, gamma = params

    if beta <= 0 or gamma <= 0:
        return 1e20

    S, I, R, incidence = simulate_sir_constant(beta, gamma, N, S0, I0, R0, t_data)
    return np.mean((incidence - y_data) ** 2)

result_constant = minimize(
    loss_constant,
    x0=[0.4, 1/7],
    bounds=[(1e-4, 2.0), (1e-4, 1.0)],
    method="L-BFGS-B"
)

beta_hat, gamma_hat = result_constant.x
S_c, I_c, R_c, inc_c = simulate_sir_constant(beta_hat, gamma_hat, N, S0, I0, R0, t_data)

mse_constant = np.mean((inc_c - y_data) ** 2)
rmse_constant = np.sqrt(mse_constant)

print("Constant-beta SIR")
print("beta =", beta_hat)
print("gamma =", gamma_hat)
print("R0 =", beta_hat / gamma_hat)
print("MSE =", mse_constant)
print("RMSE =", rmse_constant)

In [ ]:
def sir_model_timevarying(t, y, beta0, k, gamma, N):
    S, I, R = y
    beta_t = beta0 * np.exp(-k * t)

    dSdt = -beta_t * S * I / N
    dIdt = beta_t * S * I / N - gamma * I
    dRdt = gamma * I

    return [dSdt, dIdt, dRdt]

def simulate_sir_timevarying(beta0, k, gamma, N, S0, I0, R0, t_eval):
    y0 = [S0, I0, R0]

    sol = solve_ivp(
        fun=lambda t, y: sir_model_timevarying(t, y, beta0, k, gamma, N),
        t_span=(t_eval[0], t_eval[-1]),
        y0=y0,
        t_eval=t_eval,
        method="RK45"
    )

    S = sol.y[0]
    I = sol.y[1]
    R = sol.y[2]

    beta_path = beta0 * np.exp(-k * t_eval)
    incidence = beta_path * S * I / N

    return S, I, R, incidence, beta_path

def loss_timevarying(params):
    beta0, k, gamma = params

    if beta0 <= 0 or k < 0 or gamma <= 0:
        return 1e20

    S, I, R, incidence, beta_path = simulate_sir_timevarying(
        beta0, k, gamma, N, S0, I0, R0, t_data
    )

    return np.mean((incidence - y_data) ** 2)

result_tv = minimize(
    loss_timevarying,
    x0=[0.8, 0.02, 1/7],
    bounds=[(1e-4, 2.0), (0.0, 0.2), (1e-4, 1.0)],
    method="L-BFGS-B"
)

beta0_hat, k_hat, gamma_tv_hat = result_tv.x
S_tv, I_tv, R_tv, inc_tv, beta_path_tv = simulate_sir_timevarying(
    beta0_hat, k_hat, gamma_tv_hat, N, S0, I0, R0, t_data
)

mse_tv = np.mean((inc_tv - y_data) ** 2)
rmse_tv = np.sqrt(mse_tv)

print("Time-varying beta SIR")
print("beta0 =", beta0_hat)
print("k =", k_hat)
print("gamma =", gamma_tv_hat)
print("Initial R0 =", beta0_hat / gamma_tv_hat)
print("MSE =", mse_tv)
print("RMSE =", rmse_tv)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df_sir["date"], y_data, label="Observed", linewidth=2)
plt.plot(df_sir["date"], inc_c, label="Constant-beta SIR")
plt.plot(df_sir["date"], inc_tv, label="Time-varying beta SIR")
plt.title("Model comparison")
plt.xlabel("Date")
plt.ylabel("Cases")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(df_sir["date"], beta_path_tv)
plt.title("Estimated time-varying transmission rate beta(t)")
plt.xlabel("Date")
plt.ylabel("beta(t)")
plt.grid()
plt.show()

In [ ]:
summary = pd.DataFrame({
    "model": ["Constant-beta SIR", "Time-varying beta SIR"],
    "MSE": [mse_constant, mse_tv],
    "RMSE": [rmse_constant, rmse_tv]
})

summary